In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#DataFrames con la data con la cual se va a trabajar, incluye filtros y campos

#DataFrame movies filtrado por el año 200 en adelante
movies_df = spark.read.table("movie_silver.movies")
movies_df = movies_df.filter(
                            (col("year_release_date") >= 2010)
                           )\
                     .select(movies_df.movie_id, movies_df.title, movies_df.budget, movies_df.revenue, movies_df.duration_time, movies_df.release_date)

#DataFrame country
production_country_df = spark.read.table("movie_silver.productions_countries")
country_df = spark.read.table("movie_silver.countries")

#DataFrame company
movie_company_df = spark.read.table("movie_silver.movies_companies")
production_company_df = spark.read.table("movie_silver.productions_companies")



In [0]:
#DF country
country_final_df = country_df.join(production_country_df,
                                   country_df.country_id == production_country_df.country_id
                                   , "inner")\
                              .select(production_country_df.movie_id, country_df.country_name)


#DF company
company_final_df = movie_company_df.join(production_company_df,
                                         movie_company_df.company_id == production_company_df.company_id
                                         , "inner")\
                                    .select(movie_company_df.movie_id, production_company_df.company_name)

#df movies y country
movies_country_df = movies_df.join(country_final_df,
                                   movies_df.movie_id == country_final_df.movie_id
                                   , "inner")\
                               .select(movies_df["*"], country_final_df.country_name)

#df movies y production
movies_country_company_df = movies_country_df.join(company_final_df,
                                                   movies_country_df.movie_id == company_final_df.movie_id
                                                   , "inner")\
                                              .select(movies_country_df["*"], company_final_df.company_name)


In [0]:
#Seleccionamos las columnas y ordenamos

results_country_prod_company_df = movies_country_company_df.select( movies_country_company_df.title,
                                                                    movies_country_company_df.budget,
                                                                    movies_country_company_df.revenue,
                                                                    movies_country_company_df.duration_time,
                                                                    movies_country_company_df.release_date,
                                                                    movies_country_company_df.country_name,
                                                                    movies_country_company_df.company_name
                                                                )\
                                                            .orderBy(movies_country_company_df.title.desc())


results_country_prod_company_df = add_ingestion_date(results_country_prod_company_df)
results_country_prod_company_df = add_env(results_country_prod_company_df)

display(results_country_prod_company_df)

In [0]:
#Guardamos en la capa gold 
results_country_prod_company_df.write.mode("overwrite").format("delta").saveAsTable("movie_gold.results_country_prod_company")




#df = spark.read.parquet(f"{gold_folder_path}/results_country_prod_company")
#display(df)